In [2]:
import pandas as pd
import numpy as np

customers = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/datasets-cleaned/customers_cleaned.parquet"
)

articles = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/datasets-cleaned/articles_cleaned.parquet"
)

transactions = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/datasets-cleaned/transactions_cleaned.parquet"
)

customer_features = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/datasets-cleaned/customer_features.parquet"
)

article_features = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/datasets-cleaned/article_features.parquet"
)

print("Datasets loaded!")

Datasets loaded!


In [3]:
#How often customer buys
transactions = transactions.sort_values(
    ['customer_id', 't_dat']
)

transactions[
    'previous_purchase_date'
] = (
    transactions
    .groupby('customer_id')
    ['t_dat']
    .shift(1)
)

transactions[
    'days_between_purchases'
] = (
    transactions['t_dat']
    -
    transactions[
        'previous_purchase_date'
    ]
).dt.days

In [4]:
#Average purchase interval
purchase_interval = (
    transactions
    .groupby('customer_id')
    [
        'days_between_purchases'
    ]
    .mean()
    .reset_index()
)

purchase_interval.columns = [
    'customer_id',
    'avg_days_between_purchases'
]

purchase_interval.head()

,customer_id,avg_days_between_purchases
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,34.333333
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,8.519481
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,51.857143
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.000000
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,55.833333


In [5]:
#What customer buys most
transactions_articles = (
    transactions.merge(
        articles[
            [
                'article_id',
                'product_type_name'
            ]
        ],
        on='article_id',
        how='left'
    )
)

In [6]:
favorite_category = (
    transactions_articles
    .groupby(
        [
            'customer_id',
            'product_type_name'
        ]
    )
    .size()
    .reset_index(
        name='purchase_count'
    )
)

favorite_category = (
    favorite_category
    .sort_values(
        [
            'customer_id',
            'purchase_count'
        ],
        ascending=False
    )
    .drop_duplicates(
        subset=['customer_id']
    )
)

favorite_category.head()

,customer_id,product_type_name,purchase_count
10383378,ffffd9ac14e89946416d80e791d064701994755c3ab686...,Boots,1
10383373,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,Dress,2
10383371,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,Trousers,11
10383353,ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab5...,T-shirt,11
10383330,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,Swimwear bottom,8


In [7]:
#Seasonal preference
def season(month):

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Spring"

    elif month in [6, 7, 8]:
        return "Summer"

    else:
        return "Autumn"

In [8]:
transactions['season'] = (
    transactions['t_dat']
    .dt.month
    .apply(season)
)

In [9]:
season_preference = (
    transactions
    .groupby(
        [
            'customer_id',
            'season'
        ]
    )
    .size()
    .reset_index(
        name='purchase_count'
    )
)

season_preference = (
    season_preference
    .sort_values(
        [
            'customer_id',
            'purchase_count'
        ],
        ascending=False
    )
    .drop_duplicates(
        subset=['customer_id']
    )
)

season_preference.head()

,customer_id,season,purchase_count
3078773,ffffd9ac14e89946416d80e791d064701994755c3ab686...,Winter,1
3078771,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,Spring,6
3078770,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,Winter,22
3078765,ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab5...,Spring,32
3078761,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,Spring,25


In [10]:
#Online/offline tendency
channel_preference = (
    transactions
    .groupby(
        [
            'customer_id',
            'sales_channel_id'
        ]
    )
    .size()
    .reset_index(
        name='purchase_count'
    )
)

channel_preference = (
    channel_preference
    .sort_values(
        [
            'customer_id',
            'purchase_count'
        ],
        ascending=False
    )
    .drop_duplicates(
        subset=['customer_id']
    )
)

channel_preference.head()

,customer_id,sales_channel_id,purchase_count
1845535,ffffd9ac14e89946416d80e791d064701994755c3ab686...,2,1
1845534,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,2,6
1845532,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,2,32
1845530,ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab5...,2,54
1845528,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,2,32


In [16]:
repeat_purchase = (
    transactions
    .groupby(
        [
            'customer_id',
            'article_id'
        ]
    )
    .size()
    .reset_index(
        name='purchase_count'
    )
)

In [17]:
repeat_ratio = (
    repeat_purchase
    .groupby('customer_id')
    [
        'purchase_count'
    ]
    .apply(
        lambda x:
        (x > 1).mean()
    )
    .reset_index(
        name='repeat_purchase_ratio'
    )
)

repeat_ratio.head()

,customer_id,repeat_purchase_ratio
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.000000
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.187500
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.071429
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.000000
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0.083333


In [18]:
customer_behavior_features = (
    customer_features

    .merge(
        purchase_interval,
        on='customer_id',
        how='left'
    )

    .merge(
        favorite_category[
            [
                'customer_id',
                'product_type_name'
            ]
        ],
        on='customer_id',
        how='left'
    )

    .merge(
        season_preference[
            [
                'customer_id',
                'season'
            ]
        ],
        on='customer_id',
        how='left'
    )

    .merge(
        channel_preference[
            [
                'customer_id',
                'sales_channel_id'
            ]
        ],
        on='customer_id',
        how='left'
    )

    .merge(
        repeat_ratio,
        on='customer_id',
        how='left'
    )
)

customer_behavior_features.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code,purchase_frequency,days_since_last_purchase,avg_spend,avg_days_between_purchases,product_type_name,season,sales_channel_id,repeat_purchase_ratio
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,19.0,17.0,0.028628,34.333333,Blazer,Autumn,2.0,0.000000
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.0,0.0,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,78.0,76.0,0.030926,8.519481,Bikini top,Summer,2.0,0.187500
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,0.0,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,15.0,7.0,0.040435,51.857143,Sweater,Spring,2.0,0.071429
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.0,0.0,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...,2.0,471.0,0.030492,0.000000,Bra,Summer,2.0,0.000000
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...,13.0,41.0,0.036130,55.833333,Underwear body,Summer,2.0,0.083333


In [21]:
customer_behavior_features[
    'product_type_name'
] = (
    customer_behavior_features[
        'product_type_name'
    ]
    .fillna("Unknown")
    .astype(str)
)

customer_behavior_features[
    'season'
] = (
    customer_behavior_features[
        'season'
    ]
    .fillna("Unknown")
    .astype(str)
)

In [22]:
numeric_cols = [
    'avg_days_between_purchases',
    'sales_channel_id',
    'repeat_purchase_ratio'
]

customer_behavior_features[
    numeric_cols
] = customer_behavior_features[
    numeric_cols
].fillna(0)

In [23]:
customer_behavior_features.to_parquet(
    "/kaggle/working/customer_behavior_features.parquet",
    index=False
)

print("Behavioral features saved!")

Behavioral features saved!


In [24]:
customer_behavior_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371980 entries, 0 to 1371979
Data columns (total 15 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   customer_id                 1371980 non-null  object 
 1   FN                          1371980 non-null  float64
 2   Active                      1371980 non-null  float64
 3   club_member_status          1371980 non-null  object 
 4   fashion_news_frequency      1371980 non-null  object 
 5   age                         1371980 non-null  float64
 6   postal_code                 1371980 non-null  object 
 7   purchase_frequency          1371980 non-null  float64
 8   days_since_last_purchase    1371980 non-null  float64
 9   avg_spend                   1371980 non-null  float64
 10  avg_days_between_purchases  1371980 non-null  float64
 11  product_type_name           1371980 non-null  object 
 12  season                      1371980 non-null  object 
 1